# Appendix simulation: model and basis variation

This notebook compares RKHS, polynomial basis, random-forest leaf basis, random Fourier features, and nearest-neighbor matching.  GRR models estimate both ATE and ATT.  Nearest-neighbor matching is retained as an ATE-only baseline when ATT is unsupported by the matching API.

In [ ]:
from pathlib import Path
import sys
import os
import warnings
warnings.filterwarnings("once")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Make the experiment helper importable whether Jupyter is launched from the
# repository root or from notebooks/experiments.
for _candidate in [Path.cwd(), Path.cwd() / "notebooks" / "experiments"]:
    if (_candidate / "grr_experiment_utils.py").exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))

# Local experiment helpers. These live beside the notebooks and do not modify src/genriesz.
from grr_experiment_utils import *
from genriesz import (
    grr_ate, grr_att, ATEFunctional, ATTFunctional,
    BregmanGenerator, SquaredGenerator, UKLGenerator, BKLGenerator, BPGenerator,
    PolynomialBasis, TreatmentInteractionBasis,
)

# Optional random-forest leaf basis used in model-comparison experiments.
try:
    from sklearn.ensemble import RandomForestRegressor
    from genriesz.sklearn_basis import RandomForestLeafBasis
    SKLEARN_AVAILABLE = True
except Exception as exc:
    SKLEARN_AVAILABLE = False
    print("scikit-learn is not available; random-forest basis cells will fall back to RKHS.", exc)

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

In [ ]:
FAST_MODE = True
POLYNOMIAL_DEGREE = 1 if FAST_MODE else 2
DOWNLOAD_DATA = False if FAST_MODE else True
os.environ["GRR_ALLOW_REMOTE_DATA"] = "1" if DOWNLOAD_DATA else "0"

# Every analysis estimates both targets when the wrapper supports them.
# If one target fails for a particular method, the failure row is kept and the other target is displayed.
ESTIMANDS = ("ate", "att")

N_REPS = 1 if FAST_MODE else 200
N = 40 if FAST_MODE else 3000
FOLDS = 2 if FAST_MODE else 5
MAX_ITER = 25 if FAST_MODE else 500
N_FEATURES = 4 if FAST_MODE else 120

LOSS_GRID = [("SQ", None), ("UKL", None), ("BKL", None), ("BP", 0.5)]
LAMBDA_MAIN = 1e-2
ESTIMATORS_ALL = ("ra", "rw", "arw", "tmle")
print(display_mode_banner(FAST_MODE))
MODEL_GRID_APPENDIX = ["rkhs", "polynomial", "rf", "rff", "nn_matching"]
TABLE_TITLE_MODEL = "Appendix Table A3. Model and basis variation for ATE and ATT."
FIGURE_TITLE_MODEL = "Model and basis variation: ARW squared error"

In [ ]:
def dgp_factory(name, *, n, seed):
    """Three DGPs used throughout the main simulation study."""
    if name == "DGP1 nonlinear heterogeneous":
        return make_ate_data(n=n, d=8, kappa=1.0, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP2 weak overlap":
        return make_ate_data(n=n, d=8, kappa=2.5, heterogeneous=True, seed=seed, noise_sd=1.0)
    if name == "DGP3 Kang-Schafer misspecification":
        return make_kang_schafer_data(n=n, seed=seed, tau=1.0)
    raise ValueError(name)


def basis_for_model(model, *, seed=0, n_features=N_FEATURES, sigma=1.0):
    """Editable basis map used by the GRR experiments."""
    model = str(model).lower()
    if model in {"rkhs", "gaussian"}:
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"poly", "polynomial"}:
        return make_treatment_basis("poly", degree=POLYNOMIAL_DEGREE, n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rff", "fourier", "random_fourier"}:
        return make_treatment_basis("rff", n_features=n_features, sigma=sigma, seed=seed)
    if model in {"rf", "random_forest"}:
        if SKLEARN_AVAILABLE:
            rf = RandomForestRegressor(n_estimators=8 if FAST_MODE else 80, max_depth=3 if FAST_MODE else 5,
                                       min_samples_leaf=5, random_state=seed)
            return TreatmentInteractionBasis(base_basis=RandomForestLeafBasis(rf, include_bias=True, normalize=True))
        return make_treatment_basis("rkhs", n_features=n_features, sigma=sigma, seed=seed)
    raise ValueError(model)


def loss_label(loss, omega=None):
    return loss if omega is None else f"{loss}({omega:g})"


def add_metadata(df, **kwargs):
    out = df.copy()
    for k, v in kwargs.items():
        out[k] = v
    return out


def clean_results(df):
    if "status" in df.columns:
        status = df["status"].fillna("ok")
    else:
        status = pd.Series("ok", index=df.index)
    return df[status.eq("ok") & df["estimator"].ne("failed")].copy()


def mc_table(df, group_cols, estimator_filter=None):
    d = clean_results(df)
    if estimator_filter is not None:
        d = d[d["estimator"].isin(list(estimator_filter))]
    return summarize_mc(d, group_cols)


def boxplot_metric(df, *, group_col, metric, title, estimator="arw", rotate=45, ylim=None):
    d = clean_results(df)
    if estimator is not None:
        d = d[d["estimator"] == estimator]
    labels = list(d[group_col].dropna().astype(str).unique())
    data = [d.loc[d[group_col].astype(str) == lab, metric].dropna().to_numpy() for lab in labels]
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(labels)), 4.5))
    ax.boxplot(data, labels=labels, showmeans=True)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=rotate)
    if ylim is not None:
        ax.set_ylim(*ylim)
    fig.tight_layout()
    plt.show()
    return fig, ax


def run_one(data, *, estimand, loss, omega, basis, lam, cross_fit, folds, estimators=ESTIMATORS_ALL,
            penalty="l2", max_iter=MAX_ITER):
    """Fit ATE or ATT. Failures are returned as rows so tables remain complete."""
    try:
        df = fit_grr_estimand(
            data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=lam,
            cross_fit=cross_fit, folds=folds, estimators=estimators, penalty=penalty,
            max_iter=max_iter,
        )
        df["status"] = "ok"
        return df
    except Exception as exc:
        theta = true_theta_for_estimand(data, estimand)
        row = {
            "estimand": estimand.upper(), "estimator": "failed", "status": type(exc).__name__,
            "message": str(exc)[:240], "true_theta": theta,
        }
        return pd.DataFrame([row])

In [ ]:
def fit_matching_ate(data, *, M=1):
    """Nearest-neighbor matching baseline. Currently kept as ATE-only."""
    basis = PolynomialBasis(degree=1, include_bias=True)
    res = grr_ate(
        X=data["X"], Y=data["Y"], basis=basis, riesz_method="nn_matching",
        M=M, cross_fit=False, outcome_models="none", estimators=("rw",),
    )
    df = result_to_frame(res, true_theta=true_theta_for_estimand(data, "ate"))
    df["estimand"] = "ATE"
    df["status"] = "ok"
    return df

rows = []
for rep in range(N_REPS):
    data = make_ate_data(n=N, d=8, kappa=1.5, heterogeneous=True, seed=6100 + rep)
    for model_name in MODEL_GRID_APPENDIX:
        if model_name == "nn_matching":
            try:
                df = fit_matching_ate(data, M=1)
                df = add_metadata(df, rep=rep, model=model_name, loss="NN matching", cross_fit=False)
            except Exception as exc:
                df = pd.DataFrame([{"estimand": "ATE", "rep": rep, "model": model_name, "loss": "NN matching",
                                    "estimator": "failed", "status": type(exc).__name__, "message": str(exc)[:240]}])
            rows.append(df)
            continue

        for estimand in ESTIMANDS:
            for loss, omega in LOSS_GRID:
                basis = basis_for_model(model_name, seed=rep, n_features=N_FEATURES)
                df = run_one(data, estimand=estimand, loss=loss, omega=omega, basis=basis, lam=LAMBDA_MAIN,
                             cross_fit=True, folds=FOLDS, estimators=("rw", "arw"), max_iter=MAX_ITER)
                df = add_metadata(df, rep=rep, model=model_name, loss=loss_label(loss, omega), cross_fit=True)
                rows.append(df)

model_results = pd.concat(rows, ignore_index=True)
print(TABLE_TITLE_MODEL)
model_table = mc_table(model_results, ["estimand", "model", "loss", "estimator"])
display(safe_display_frame(model_table, n=150))

In [ ]:
plot_df = clean_results(model_results).copy()
plot_df = plot_df[plot_df["estimator"].isin(["rw", "arw"])]
plot_df["method"] = plot_df["estimand"] + " | " + plot_df["model"] + " | " + plot_df["loss"] + " | " + plot_df["estimator"]
boxplot_metric(plot_df, group_col="method", metric="squared_error", title=FIGURE_TITLE_MODEL, estimator=None, rotate=80)